# DPO preference tuning of Qwen2.5 on forecasting questions (experiment `dpo-forecast`)

Part of the **Experiments lab** of Forecast Arena (`docs/project/LAB_PLAN.md`). This notebook is an *evaluation*, not the live
record. It follows the pre-registration in `lab/prereg/dpo-forecast.md`; do not change the settings below after seeing
test results.

**How to run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. It takes roughly 1.5 to 2.5 hours.
At the end it prints a table of SHA-256 values and downloads `outputs.zip`. Then follow `docs/project/HUMAN_TODO.md` item H35
(attach the zip and the GGUF to a GitHub release and paste the hashes into the chat).

What it does: scores the untuned model (the DPO reference), trains a 4-bit LoRA adapter with the Direct Preference Optimisation
loss (the preferred answer is the one that matches the outcome), scores the tuned model on the validation and the two frozen test
windows, saves the adapter, scores and logs, and exports a GGUF with a loud, checked export step. Test windows are never used for
training or calibration. The comparison that matters is with the QLoRA run (`lab-qlora-r1`) at the same data and budget.


In [ ]:
# Settings (pre-registered; change only TRAIN_ROWS / switches if the session is too short)
TRAIN_ROWS = 2000    # rows of the committed train file (seeded, template-proportional subsample)
BETA = 0.1           # DPO temperature (pre-registered)
EXPORT_GGUF = True   # export the tuned model as GGUF Q4_K_M (needs a few minutes of compiling)
BRANCH = "main"


In [ ]:
import os, subprocess, sys
print("python", sys.version.split()[0])
!pip -q install "transformers==5.22.2" "peft==0.22.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "scikit-learn==1.9.1" huggingface_hub "pydantic>=2.13,<3" "click>=8.1,<9" "jinja2>=3.1,<4" "numpy>=2,<3" "anthropic>=2.8,<3" "duckdb>=1.1,<2"
def clone(branch):
    !rm -rf /content/forecast-arena && git clone -q --depth 1 --branch $branch https://github.com/JASHD12/forecast-arena.git /content/forecast-arena
clone(BRANCH)
if not os.path.exists("/content/forecast-arena/lab/data/forecast-sft/manifest.json"):
    print("the lab data is not on", BRANCH, "yet (its pull request is not merged); using claude/experiments-lab")
    clone("claude/experiments-lab")
os.chdir("/content/forecast-arena")
sys.path.insert(0, "/content/forecast-arena/src")  # no editable install: it needs Python 3.12 and a restart
import arena, arena.lab
print("arena from", arena.__file__)

In [ ]:
import hashlib, json, os, platform, subprocess, sys, time
from pathlib import Path
import torch

os.environ["HF_HUB_DISABLE_XET"] = "1"
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again"
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| torch", torch.__version__, "| python", platform.python_version())

In [ ]:
# 1. Data: the committed, hashed splits (nothing to upload)
from arena.lab import dataset as ds

root = Path("lab/data/forecast-sft")
manifest = json.loads((root / "manifest.json").read_text())
rows = {}
for name, info in manifest["splits"].items():
    assert ds.plain_sha256(root / f"{name}.jsonl.gz") == info["sha256"], f"hash mismatch in {name}"
    rows[name] = ds.read_split(root / f"{name}.jsonl.gz")
print({k: len(v) for k, v in rows.items()})
PREREG_SHA = hashlib.sha256(Path("lab/prereg/dpo-forecast.md").read_bytes()).hexdigest()
print("prereg sha256:", PREREG_SHA)

In [ ]:
# 2. One function per variant (zero-shot scores, training, tuned scores, files)
from transformers import AutoTokenizer
from arena.lab.finetune import train as tr
from arena.lab.finetune.load import add_lora, load_base
from arena.lab.finetune.run import Spec, file_hashes, run_variant

OUT = Path("/content/outputs"); OUT.mkdir(exist_ok=True)
VARIANTS = {  # name: (base model, 4-bit?, untuned variant name, pinned revision)
    "dpo_1_5b": ("Qwen/Qwen2.5-1.5B-Instruct", True, "base_1_5b", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"),
}
commits = {}

def go(name):
    base_id, four_bit, base_name, rev = VARIANTS[name]
    tok = AutoTokenizer.from_pretrained(base_id, revision=rev)
    cfg = tr.TrainConfig(fp16=True, log_every=5)

    def load():
        m = load_base(base_id, revision=rev, four_bit=four_bit)
        if not four_bit:
            m = m.to("cuda")
        commits[name] = getattr(m.config, "_commit_hash", None)
        return m

    out = run_variant(
        Spec(name, base_id, four_bit, base_name, dpo_beta=BETA), rows, tok, load,
        lambda m: add_lora(m, four_bit=four_bit), OUT, cfg,
        device=torch.device("cuda"), train_rows=TRAIN_ROWS, eval_batch=4, on_log=print,
    )
    torch.cuda.empty_cache()
    return out

In [ ]:
# 3. The run: DPO on Qwen2.5-1.5B-Instruct, reference = the untuned model (about 1 to 2 hours on a T4)
summaries = {"dpo_1_5b": go("dpo_1_5b")}
print(json.dumps(summaries["dpo_1_5b"], indent=1))


In [ ]:
# 5. Optional: merge the 1.5B adapter and export GGUF Q4_K_M (scored on CPU later, in GitHub Actions)
# Every step must succeed (a failing `!` line would otherwise be ignored) and the file must exist.
import shutil, subprocess, sys
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd} -> exit {r.returncode}: {(r.stdout + r.stderr)[-600:]}")
    return r.stdout.strip()
gguf = {}
if EXPORT_GGUF:
    try:
        from peft import PeftModel
        from transformers import AutoModelForCausalLM
        from huggingface_hub import hf_hub_download
        base_id, rev = VARIANTS["dpo_1_5b"][0], VARIANTS["dpo_1_5b"][3]
        base = AutoModelForCausalLM.from_pretrained(base_id, revision=rev, torch_dtype=torch.float16)
        merged = PeftModel.from_pretrained(base, str(OUT / "adapter_dpo_1_5b")).merge_and_unload()
        merged.save_pretrained("/content/merged_1_5b")
        # The converter needs an older transformers than the one that saved the tokenizer, so the
        # original tokenizer files of the pinned revision are copied instead of re-saved.
        for name in ("tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt"):
            shutil.copy(hf_hub_download(base_id, name, revision=rev), f"/content/merged_1_5b/{name}")
        del base, merged; gc.collect()
        shutil.rmtree("/content/llama.cpp", ignore_errors=True)
        sh("git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp")
        gguf["llama_cpp_commit"] = sh("git -C /content/llama.cpp rev-parse HEAD")
        # converter requirements in their own environment: they pin a different transformers
        sh(f"{sys.executable} -m venv /content/conv && /content/conv/bin/pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt")
        sh("/content/conv/bin/python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_1_5b --outfile /content/qwen2.5-1.5b-dpo-f16.gguf --outtype f16")
        sh("cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF")
        sh("cmake --build /content/llama.cpp/build --target llama-quantize -j 2")
        target = OUT / "qwen2.5-1.5b-dpo-q4_k_m.gguf"
        sh(f"/content/llama.cpp/build/bin/llama-quantize /content/qwen2.5-1.5b-dpo-f16.gguf {target} Q4_K_M")
        assert target.exists() and target.stat().st_size > 5e8, "quantised GGUF missing or too small"
        gguf["exported"] = True
        gguf["size_bytes"] = target.stat().st_size
    except Exception as e:  # the adapter and scores are the main results; the GGUF is a bonus
        gguf["exported"] = False; gguf["error"] = repr(e)[-600:]
        print("GGUF export failed (the rest of the run is fine):", e)
print(gguf)


In [ ]:
# 6. Summary, hashes and download
import importlib.metadata as md
def ver(p):
    try: return md.version(p)
    except Exception: return None
summary = {
    "experiment": "dpo-forecast", "prereg_sha256": PREREG_SHA,
    "dataset_sha256": {k: v["sha256"] for k, v in manifest["splits"].items()},
    "gpu": GPU, "train_rows": TRAIN_ROWS, "variants": summaries, "model_commits": commits, "gguf": gguf,
    "packages": {p: ver(p) for p in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "scikit-learn"]},
    "code_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
(OUT / "run_summary.json").write_text(json.dumps(summary, indent=1))
hashes = file_hashes(OUT)
(OUT / "SHA256SUMS.json").write_text(json.dumps(hashes, indent=1))
for k, v in hashes.items():
    print(v, k)
!cd /content && zip -qr outputs.zip outputs
print("sha256 of outputs.zip:", hashlib.sha256(Path("/content/outputs.zip").read_bytes()).hexdigest())
try:
    from google.colab import files
    files.download("/content/outputs.zip")
except Exception as e:
    print("download manually from the file browser:", e)